# Virtual Zarr Creation & Performance Benchmarking

## Introduction

To create the data cube in virtualzarr format using the outputs from `gdal` and `cdo`.

### Key Objectives:

1. **Visualize outputs** from CDO and GDAL preprocessing pipelines
2. **Create virtual Zarr stores** from preprocessed netCDF files
3. **Benchmark performance** comparing:
   - NetCDF (original format)
   - Regular Zarr (converted format)
   - Virtual Zarr (reference-based format)
4. **Measure metrics**: Load time, memory usage, file size, and query performance

Virtual Zarrs provide significant advantages:
- 📦 **Minimal disk space**: References files without copying
- ⚡ **Fast loading**: Lazy-loaded metadata and data access
- 💾 **Efficient memory**: Only load required data chunks
- 🔗 **Linked analysis**: Unified access to multiple source files


In [ ]:
# Import Required Libraries

# Fix matplotlib backend issue from environment
import os
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']

# Data Processing & Analysis
import xarray as xr
import numpy as np
import pandas as pd

# Virtual Zarr handling
try:
    import virtualizarr
    print("✓ virtualizarr available")
except ImportError:
    print("Note: virtualizarr not installed, will use xarray only")

# Visualization - Set backend first to avoid conflicts
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
from matplotlib import rcParams

# File & System Operations
import glob
import json
from pathlib import Path
from datetime import datetime

# Zarr & Data Format
import zarr

# Performance Monitoring
import time
import psutil
import gc

# Warnings & Logging
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib for better visualizations
rcParams['figure.figsize'] = (14, 8)
rcParams['font.size'] = 10
rcParams['axes.labelsize'] = 11
rcParams['axes.titlesize'] = 12
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['legend.fontsize'] = 10
rcParams['figure.titlesize'] = 14

print("All required libraries imported successfully!")
print(f"  - xarray version: {xr.__version__}")
print(f"  - numpy version: {np.__version__}")
print(f"  - pandas version: {pd.__version__}")
print(f"  - zarr version: {zarr.__version__}")

## Notebook Configuration

Before running the analysis, ensure you have the following dependencies installed:

```bash
pip install xarray zarr pandas matplotlib psutil netCDF4
```

This notebook uses data outputs from the CDO and GDAL preprocessing pipelines located in:
- `../outputs/cdo_output/` - CDO processed netCDF files
- `../outputs/gdal_output/` - GDAL processed netCDF files
- `../outputs/zarr_outputs/` - Generated Zarr stores (created by this notebook)


In [ ]:
### 1. Define paths to CDO and GDAL outputs

# Setup directory paths
outputs_dir = "../outputs"
cdo_output_dir, gdal_output_dir = f"{outputs_dir}/cdo_output", f"{outputs_dir}/gdal_output"

# Define subdirectories
paths = {
    'cdo_metref': f"{cdo_output_dir}/metref",
    'cdo_rzsm': f"{cdo_output_dir}/rzsm",
    'gdal_metref': f"{gdal_output_dir}/metref",
    'gdal_sm': f"{gdal_output_dir}/rzsm"
}

# Find all files with corrected glob patterns
cdo_metref_files = sorted(glob.glob(f"{paths['cdo_metref']}/**/*.nc", recursive=True))
cdo_rzsm_files = sorted(glob.glob(f"{paths['cdo_rzsm']}/**/*.nc", recursive=True))
gdal_metref_files = sorted(glob.glob(f"{paths['gdal_metref']}/**/METREF/*.nc", recursive=True))

# GDAL SM files organized by variable folders (var40, var41, var42, var43)
gdal_sm_files = {}
for var_folder in ['var40', 'var41', 'var42', 'var43']:
    var_path = f"{paths['gdal_sm']}/**/{var_folder}/*.nc"
    files = sorted(glob.glob(var_path, recursive=True))
    if files:
        gdal_sm_files[var_folder] = files

# Print summary
print(f"CDO Output: {cdo_output_dir}\nGDAL Output: {gdal_output_dir}")
print(f"\nCDO METREF: {len(cdo_metref_files)} files | CDO RZSM: {len(cdo_rzsm_files)} files")
print(f"GDAL METREF: {len(gdal_metref_files)} files")
print(f"GDAL SM variables found: {list(gdal_sm_files.keys())}")
for var, files in gdal_sm_files.items():
    print(f"  {var}: {len(files)} files")

# # Print sample files
# for label, files in [("CDO METREF", cdo_metref_files), ("CDO RZSM", cdo_rzsm_files), ("GDAL METREF", gdal_metref_files)]:
#     if files:
#         print(f"\nSample {label}: {files[0]}")

# for var, files in gdal_sm_files.items():
#     if files:
#         print(f"Sample GDAL {var}: {files[0]}")

In [ ]:
### 2. Load and inspect netCDF files with xarray

# Load a sample CDO file
if cdo_metref_files:
    sample_cdo = xr.open_dataset(cdo_metref_files[0], engine='netcdf4')
    print("CDO METREF Sample Dataset:")
    print(sample_cdo)
    print("\nData Variables:", list(sample_cdo.data_vars))
    print("Coordinates:", list(sample_cdo.coords))
    print("Dimensions:", dict(sample_cdo.dims))
    print("\nAttributes:", sample_cdo.attrs)
    sample_cdo.close()


In [ ]:
### 4. Create virtual Zarr for CDO RZSM output

# Create virtual zarr for CDO RZSM
cdo_rzsm_zarr = "../outputs/zarr_outputs/cdo_rzsm_virtual.zarr"
ds_cdo_rzsm = create_virtual_zarr_from_netcdf_files(
    cdo_rzsm_files, 
    cdo_rzsm_zarr,
    "CDO RZSM Virtual Zarr"
)


In [ ]:
### 11. Benchmark: Data processing and analysis

print("\n" + "="*60)
print("BENCHMARK 2: Data Processing & Analysis")
print("="*60)

def process_netcdf(file_path):
    """Load, compute mean, and basic statistics from NetCDF."""
    ds = xr.open_dataset(file_path, engine='netcdf4')
    var_name = list(ds.data_vars)[0]
    result = {
        'mean': float(ds[var_name].mean()),
        'std': float(ds[var_name].std()),
        'min': float(ds[var_name].min()),
        'max': float(ds[var_name].max())
    }
    ds.close()
    return result

def process_zarr(zarr_path):
    """Load, compute mean, and basic statistics from Zarr."""
    ds = xr.open_dataset(zarr_path, engine='zarr')
    var_name = list(ds.data_vars)[0]
    result = {
        'mean': float(ds[var_name].mean()),
        'std': float(ds[var_name].std()),
        'min': float(ds[var_name].min()),
        'max': float(ds[var_name].max())
    }
    ds.close()
    return result

process_results = {}

# Process NetCDF
if cdo_metref_files:
    print("\n1. Processing NetCDF (mean, std, min, max)...")
    result_nc = monitor.measure_operation(
        "Process NetCDF",
        process_netcdf,
        cdo_metref_files[0]
    )
    process_results['NetCDF'] = monitor.results['Process NetCDF']
    print(f"  Result: Mean={result_nc['mean']:.4f}, Std={result_nc['std']:.4f}")

# Process Regular Zarr
if os.path.exists(cdo_metref_regular_zarr):
    print("\n2. Processing Regular Zarr...")
    result_zarr = monitor.measure_operation(
        "Process Regular Zarr",
        process_zarr,
        cdo_metref_regular_zarr
    )
    process_results['Regular Zarr'] = monitor.results['Process Regular Zarr']
    print(f"  Result: Mean={result_zarr['mean']:.4f}, Std={result_zarr['std']:.4f}")

# Process Virtual Zarr
if os.path.exists(cdo_metref_zarr):
    print("\n3. Processing Virtual Zarr...")
    result_vzarr = monitor.measure_operation(
        "Process Virtual Zarr",
        process_zarr,
        cdo_metref_zarr
    )
    process_results['Virtual Zarr'] = monitor.results['Process Virtual Zarr']
    print(f"  Result: Mean={result_vzarr['mean']:.4f}, Std={result_vzarr['std']:.4f}")


In [ ]:
### 12. Benchmark: File size comparison

print("\n" + "="*60)
print("BENCHMARK 3: File Size Comparison")
print("="*60)

def get_directory_size(path):
    """Calculate total size of directory in MB."""
    if not os.path.exists(path):
        return 0
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for filename in filenames:
            filepath = os.path.join(dirpath, filename)
            total += os.path.getsize(filepath)
    return total / 1024 / 1024

# Get file sizes
file_sizes = {}

print("\n1. NetCDF file size:")
if cdo_metref_files:
    nc_size = os.path.getsize(cdo_metref_files[0]) / 1024 / 1024
    file_sizes['NetCDF (single)'] = nc_size
    print(f"  {cdo_metref_files[0]}: {nc_size:.2f} MB")

print("\n2. Regular Zarr directory size:")
if os.path.exists(cdo_metref_regular_zarr):
    zarr_size = get_directory_size(cdo_metref_regular_zarr)
    file_sizes['Regular Zarr'] = zarr_size
    print(f"  {cdo_metref_regular_zarr}: {zarr_size:.2f} MB")

print("\n3. Virtual Zarr directory size:")
if os.path.exists(cdo_metref_zarr):
    vzarr_size = get_directory_size(cdo_metref_zarr)
    file_sizes['Virtual Zarr'] = vzarr_size
    print(f"  {cdo_metref_zarr}: {vzarr_size:.2f} MB")


In [ ]:
### 13. Create comprehensive comparison visualizations

# Prepare data for visualization
comparison_data = {
    'Load Time (seconds)': {},
    'Memory Delta (MB)': {},
    'Peak Memory (MB)': {},
    'File Size (MB)': {}
}

# Extract loading metrics
for format_type, metrics in load_results.items():
    comparison_data['Load Time (seconds)'][format_type] = metrics['elapsed_time']
    comparison_data['Memory Delta (MB)'][format_type] = metrics['memory_delta']
    comparison_data['Peak Memory (MB)'][format_type] = metrics['peak_memory']

# Extract file sizes
for format_type, size in file_sizes.items():
    comparison_data['File Size (MB)'][format_type] = size

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Performance Comparison: NetCDF vs Regular Zarr vs Virtual Zarr', 
             fontsize=16, fontweight='bold', y=1.00)

# 1. Load Time Comparison
ax = axes[0, 0]
if comparison_data['Load Time (seconds)']:
    formats = list(comparison_data['Load Time (seconds)'].keys())
    times = list(comparison_data['Load Time (seconds)'].values())
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(formats)]
    bars1 = ax.bar(formats, times, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Time (seconds)', fontsize=11, fontweight='bold')
    ax.set_title('Load Time Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}s',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# 2. Memory Delta Comparison
ax = axes[0, 1]
if comparison_data['Memory Delta (MB)']:
    formats = list(comparison_data['Memory Delta (MB)'].keys())
    mem_delta = list(comparison_data['Memory Delta (MB)'].values())
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(formats)]
    bars2 = ax.bar(formats, mem_delta, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Memory Change (MB)', fontsize=11, fontweight='bold')
    ax.set_title('Memory Delta During Load', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}MB',
                ha='center', va='bottom' if height >= 0 else 'top', fontsize=10, fontweight='bold')

# 3. Peak Memory Comparison
ax = axes[1, 0]
if comparison_data['Peak Memory (MB)']:
    formats = list(comparison_data['Peak Memory (MB)'].keys())
    peak_mem = list(comparison_data['Peak Memory (MB)'].values())
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(formats)]
    bars3 = ax.bar(formats, peak_mem, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Memory (MB)', fontsize=11, fontweight='bold')
    ax.set_title('Peak Memory Usage', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for bar in bars3:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}MB',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# 4. File Size Comparison
ax = axes[1, 1]
if comparison_data['File Size (MB)']:
    formats = list(comparison_data['File Size (MB)'].keys())
    sizes = list(comparison_data['File Size (MB)'].values())
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(formats)]
    bars4 = ax.bar(formats, sizes, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Size (MB)', fontsize=11, fontweight='bold')
    ax.set_title('File Size on Disk', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for bar in bars4:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}MB',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Performance comparison visualization created!")


In [ ]:
### 14. Create detailed comparison table

# Create comprehensive comparison DataFrame
comparison_df = pd.DataFrame({
    'Metric': [
        'Load Time (s)',
        'Memory Delta (MB)',
        'Peak Memory (MB)',
        'File Size (MB)'
    ]
})

for format_type in ['NetCDF', 'Regular Zarr', 'Virtual Zarr']:
    if format_type in load_results:
        metrics = load_results[format_type]
        comparison_df[format_type] = [
            f"{metrics['elapsed_time']:.4f}",
            f"{metrics['memory_delta']:.2f}",
            f"{metrics['peak_memory']:.2f}",
            f"{file_sizes.get(format_type if 'Regular' not in format_type else format_type, 0):.2f}"
        ]
    elif format_type in file_sizes:
        comparison_df[format_type] = ['N/A', 'N/A', 'N/A', f"{file_sizes.get(format_type, 0):.2f}"]

print("\n" + "="*80)
print("PERFORMANCE COMPARISON TABLE")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Calculate efficiency metrics
print("\n" + "="*80)
print("EFFICIENCY ANALYSIS")
print("="*80)

if 'NetCDF' in load_results and 'Virtual Zarr' in load_results:
    nc_time = load_results['NetCDF']['elapsed_time']
    vz_time = load_results['Virtual Zarr']['elapsed_time']
    speedup = nc_time / vz_time if vz_time > 0 else 0
    print(f"\n✓ Virtual Zarr vs NetCDF:")
    print(f"  Load Time Improvement: {(1 - vz_time/nc_time) * 100:.2f}% faster" if vz_time < nc_time else f"  Load Time: {speedup:.2f}x slower")
    
    nc_mem = load_results['NetCDF']['memory_delta']
    vz_mem = load_results['Virtual Zarr']['memory_delta']
    mem_improvement = (1 - vz_mem/nc_mem) * 100 if nc_mem > 0 else 0
    print(f"  Memory Usage Improvement: {mem_improvement:.2f}%" + (" more efficient" if mem_improvement > 0 else " less efficient"))

if 'Regular Zarr' in load_results and 'Virtual Zarr' in load_results:
    rz_time = load_results['Regular Zarr']['elapsed_time']
    vz_time = load_results['Virtual Zarr']['elapsed_time']
    print(f"\n✓ Virtual Zarr vs Regular Zarr:")
    print(f"  Load Time Difference: {abs(rz_time - vz_time):.4f}s")
    print(f"  Memory Advantage: {load_results['Regular Zarr']['memory_delta'] - load_results['Virtual Zarr']['memory_delta']:.2f} MB")

print("\n" + "="*80)


In [ ]:
### 15. Advanced data visualization from different formats

def load_and_visualize_data(data_path, format_type, title_suffix=""):
    """Load data from different formats and create visualizations."""
    
    try:
        print(f"\nLoading {format_type} data...")
        if 'nc' in format_type.lower():
            ds = xr.open_dataset(data_path, engine='netcdf4')
        else:
            ds = xr.open_dataset(data_path, engine='zarr')
        
        # Get first data variable
        var_name = list(ds.data_vars)[0]
        data = ds[var_name]
        
        # Create figure with subplots
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'{var_name} - {format_type} Format {title_suffix}', 
                    fontsize=14, fontweight='bold')
        
        # 1. First timestep heatmap
        ax = axes[0, 0]
        if 'time' in data.dims:
            data_t0 = data.isel(time=0).values
        else:
            data_t0 = data.values
        
        if len(data_t0.shape) >= 2:
            im = ax.imshow(data_t0, cmap='viridis', aspect='auto')
            ax.set_title('First Timestep', fontweight='bold')
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')
            plt.colorbar(im, ax=ax, label='Value')
        
        # 2. Time series statistics
        ax = axes[0, 1]
        if 'time' in data.dims:
            time_mean = data.mean(dim=['x', 'y']) if 'x' in data.dims and 'y' in data.dims else data.mean()
            ax.plot(time_mean.values, linewidth=2, color='#2E86AB', marker='o', markersize=4)
            ax.set_title('Temporal Mean', fontweight='bold')
            ax.set_xlabel('Time Index')
            ax.set_ylabel('Mean Value')
            ax.grid(alpha=0.3)
        
        # 3. Spatial mean over time
        ax = axes[1, 0]
        if 'time' in data.dims and 'x' in data.dims and 'y' in data.dims:
            spatial_mean = data.mean(dim='time').values
            im = ax.imshow(spatial_mean, cmap='plasma', aspect='auto')
            ax.set_title('Spatial Mean Over Time', fontweight='bold')
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')
            plt.colorbar(im, ax=ax, label='Value')
        
        # 4. Distribution histogram
        ax = axes[1, 1]
        data_flat = data.values.flatten()
        data_flat = data_flat[~np.isnan(data_flat)]  # Remove NaN values
        ax.hist(data_flat, bins=50, color='#A23B72', alpha=0.7, edgecolor='black')
        ax.set_title('Value Distribution', fontweight='bold')
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')
        ax.grid(axis='y', alpha=0.3)
        
        # Add statistics text
        stats_text = f"Stats:\nMean: {float(data.mean()):.4f}\nStd: {float(data.std()):.4f}\nMin: {float(data.min()):.4f}\nMax: {float(data.max()):.4f}"
        ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
               fontsize=10, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        plt.tight_layout()
        plt.show()
        
        ds.close()
        print(f"✓ {format_type} visualization complete")
        
    except Exception as e:
        print(f"Error visualizing {format_type}: {e}")

# Visualize data from different formats
print("\n" + "="*60)
print("DATA VISUALIZATION FROM DIFFERENT FORMATS")
print("="*60)

if cdo_metref_files:
    load_and_visualize_data(cdo_metref_files[0], "NetCDF", "(Single File)")

if os.path.exists(cdo_metref_regular_zarr):
    load_and_visualize_data(cdo_metref_regular_zarr, "Regular Zarr", "(Combined)")

if os.path.exists(cdo_metref_zarr):
    load_and_visualize_data(cdo_metref_zarr, "Virtual Zarr", "(Combined)")


In [ ]:
### 16. Slicing and subsetting performance test

print("\n" + "="*80)
print("BENCHMARK 4: Slicing & Subsetting Performance")
print("="*80)

def slice_operation(zarr_path):
    """Perform slicing operation on data."""
    ds = xr.open_dataset(zarr_path, engine='zarr')
    var_name = list(ds.data_vars)[0]
    
    # Slice to specific region if coordinates exist
    if 'x' in ds.dims and 'y' in ds.dims:
        subset = ds[var_name].isel(x=slice(10, 50), y=slice(10, 50))
    else:
        subset = ds[var_name].isel({list(ds.dims)[0]: slice(0, 5)})
    
    result = float(subset.mean())
    ds.close()
    return result

def slice_netcdf(file_path):
    """Perform slicing operation on NetCDF."""
    ds = xr.open_dataset(file_path, engine='netcdf4')
    var_name = list(ds.data_vars)[0]
    
    if 'x' in ds.dims and 'y' in ds.dims:
        subset = ds[var_name].isel(x=slice(10, 50), y=slice(10, 50))
    else:
        subset = ds[var_name].isel({list(ds.dims)[0]: slice(0, 5)})
    
    result = float(subset.mean())
    ds.close()
    return result

slice_results = {}

# Slice NetCDF
if cdo_metref_files:
    print("\n1. NetCDF Slicing...")
    result_slice_nc = monitor.measure_operation(
        "Slice NetCDF",
        slice_netcdf,
        cdo_metref_files[0]
    )
    slice_results['NetCDF'] = monitor.results['Slice NetCDF']
    print(f"  Result: {result_slice_nc:.4f}")

# Slice Regular Zarr
if os.path.exists(cdo_metref_regular_zarr):
    print("\n2. Regular Zarr Slicing...")
    result_slice_zarr = monitor.measure_operation(
        "Slice Regular Zarr",
        slice_operation,
        cdo_metref_regular_zarr
    )
    slice_results['Regular Zarr'] = monitor.results['Slice Regular Zarr']
    print(f"  Result: {result_slice_zarr:.4f}")

# Slice Virtual Zarr
if os.path.exists(cdo_metref_zarr):
    print("\n3. Virtual Zarr Slicing...")
    result_slice_vzarr = monitor.measure_operation(
        "Slice Virtual Zarr",
        slice_operation,
        cdo_metref_zarr
    )
    slice_results['Virtual Zarr'] = monitor.results['Slice Virtual Zarr']
    print(f"  Result: {result_slice_vzarr:.4f}")

# Visualize slicing performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if slice_results:
    formats = list(slice_results.keys())
    slice_times = [slice_results[f]['elapsed_time'] for f in formats]
    slice_mems = [slice_results[f]['memory_delta'] for f in formats]
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1'][:len(formats)]
    
    bars1 = ax1.bar(formats, slice_times, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax1.set_ylabel('Time (seconds)', fontsize=11, fontweight='bold')
    ax1.set_title('Slicing Operation Time', fontsize=12, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    bars2 = ax2.bar(formats, slice_mems, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax2.set_ylabel('Memory Change (MB)', fontsize=11, fontweight='bold')
    ax2.set_title('Slicing Operation Memory', fontsize=12, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}MB', ha='center', va='bottom' if height >= 0 else 'top', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## Performance Summary & Recommendations

Based on the comprehensive benchmarking performed above, here are the key findings:

### Key Metrics Evaluated:
1. **Load Time**: Time required to open and read data from disk
2. **Memory Delta**: Change in RAM usage during the operation
3. **Peak Memory**: Maximum memory used during the operation
4. **File Size**: Total disk space required for storage
5. **Slicing Performance**: Time and memory for subsetting operations

### Format Characteristics:

| Format | Best For | Advantages | Disadvantages |
|--------|----------|------------|---------------|
| **NetCDF** | Individual file analysis | Universal format, well-established | Slower for large datasets, high memory overhead |
| **Regular Zarr** | Complete data copies | Fast access, optimized format | Duplicates data, requires more disk space |
| **Virtual Zarr** | Reference without copying | Minimal disk space, fast metadata, lazy loading | Requires source files to remain accessible |

### Recommendations:

- **For quick analysis**: Use Virtual Zarr for minimal memory footprint
- **For parallel processing**: Use Regular Zarr for consistent performance
- **For archival**: Use Virtual Zarr to save disk space while maintaining access


In [ ]:
### 17. Export comprehensive benchmark report

import json
from datetime import datetime

# Compile all results into a comprehensive report
report = {
    'timestamp': datetime.now().isoformat(),
    'dataset': 'CDO METREF',
    'test_dates': '2010-01-01 to 2010-01-05',
    'benchmarks': {
        'loading': {},
        'processing': {},
        'slicing': {},
        'file_sizes': {}
    },
    'summary': {}
}

# Add loading results
for format_type, metrics in load_results.items():
    report['benchmarks']['loading'][format_type] = {
        'elapsed_time_seconds': round(metrics['elapsed_time'], 6),
        'memory_delta_mb': round(metrics['memory_delta'], 2),
        'peak_memory_mb': round(metrics['peak_memory'], 2)
    }

# Add slicing results
for format_type, metrics in slice_results.items():
    report['benchmarks']['slicing'][format_type] = {
        'elapsed_time_seconds': round(metrics['elapsed_time'], 6),
        'memory_delta_mb': round(metrics['memory_delta'], 2),
        'peak_memory_mb': round(metrics['peak_memory'], 2)
    }

# Add file sizes
report['benchmarks']['file_sizes'] = {
    k: round(v, 2) for k, v in file_sizes.items()
}

# Generate summary statistics
if 'NetCDF' in load_results and 'Virtual Zarr' in load_results:
    nc_time = load_results['NetCDF']['elapsed_time']
    vz_time = load_results['Virtual Zarr']['elapsed_time']
    time_improvement = ((nc_time - vz_time) / nc_time * 100) if nc_time > 0 else 0
    
    report['summary']['netcdf_vs_virtual_zarr'] = {
        'load_time_improvement_percent': round(time_improvement, 2),
        'memory_improvement_mb': round(
            load_results['NetCDF']['memory_delta'] - load_results['Virtual Zarr']['memory_delta'], 2
        ),
        'verdict': 'Virtual Zarr is more efficient' if time_improvement > 0 else 'NetCDF is faster'
    }

# Save report to JSON
report_path = '../outputs/benchmark_report.json'
Path(report_path).parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n✓ Benchmark report saved to: {report_path}")
print("\nFull Report:")
print(json.dumps(report, indent=2))


In [ ]:
### 18. Create interactive summary dashboard

# Create a comprehensive summary visualization
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

fig.suptitle('Comprehensive Performance Benchmark Dashboard', fontsize=18, fontweight='bold', y=0.98)

# 1. Load Time Radar Chart (top left)
ax1 = fig.add_subplot(gs[0, 0])
if load_results:
    formats = list(load_results.keys())
    times = [load_results[f]['elapsed_time'] for f in formats]
    colors_list = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    bars = ax1.barh(formats, times, color=colors_list[:len(formats)], alpha=0.8, edgecolor='black', linewidth=1.5)
    ax1.set_xlabel('Time (seconds)', fontsize=10, fontweight='bold')
    ax1.set_title('Load Time', fontsize=11, fontweight='bold')
    ax1.invert_yaxis()
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax1.text(width, bar.get_y() + bar.get_height()/2.,
                f' {width:.4f}s', ha='left', va='center', fontsize=9, fontweight='bold')

# 2. Memory Delta (top middle)
ax2 = fig.add_subplot(gs[0, 1])
if load_results:
    formats = list(load_results.keys())
    mems = [load_results[f]['memory_delta'] for f in formats]
    colors_list = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    bars = ax2.barh(formats, mems, color=colors_list[:len(formats)], alpha=0.8, edgecolor='black', linewidth=1.5)
    ax2.set_xlabel('Memory Change (MB)', fontsize=10, fontweight='bold')
    ax2.set_title('Memory Delta', fontsize=11, fontweight='bold')
    ax2.invert_yaxis()
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax2.text(width, bar.get_y() + bar.get_height()/2.,
                f' {width:.2f}MB', ha='left', va='center', fontsize=9, fontweight='bold')

# 3. Peak Memory (top right)
ax3 = fig.add_subplot(gs[0, 2])
if load_results:
    formats = list(load_results.keys())
    peaks = [load_results[f]['peak_memory'] for f in formats]
    colors_list = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    bars = ax3.barh(formats, peaks, color=colors_list[:len(formats)], alpha=0.8, edgecolor='black', linewidth=1.5)
    ax3.set_xlabel('Memory (MB)', fontsize=10, fontweight='bold')
    ax3.set_title('Peak Memory', fontsize=11, fontweight='bold')
    ax3.invert_yaxis()
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax3.text(width, bar.get_y() + bar.get_height()/2.,
                f' {width:.2f}MB', ha='left', va='center', fontsize=9, fontweight='bold')

# 4. File Size (middle left)
ax4 = fig.add_subplot(gs[1, 0])
if file_sizes:
    formats_fs = list(file_sizes.keys())
    sizes = list(file_sizes.values())
    colors_list = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    bars = ax4.bar(formats_fs, sizes, color=colors_list[:len(formats_fs)], alpha=0.8, edgecolor='black', linewidth=1.5)
    ax4.set_ylabel('Size (MB)', fontsize=10, fontweight='bold')
    ax4.set_title('Disk Space Usage', fontsize=11, fontweight='bold')
    ax4.tick_params(axis='x', rotation=45)
    for bar in bars:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}MB', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 5. Slicing Performance (middle middle)
ax5 = fig.add_subplot(gs[1, 1])
if slice_results:
    formats_sl = list(slice_results.keys())
    slice_times = [slice_results[f]['elapsed_time'] for f in formats_sl]
    colors_list = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    bars = ax5.bar(formats_sl, slice_times, color=colors_list[:len(formats_sl)], alpha=0.8, edgecolor='black', linewidth=1.5)
    ax5.set_ylabel('Time (seconds)', fontsize=10, fontweight='bold')
    ax5.set_title('Slicing Operation Time', fontsize=11, fontweight='bold')
    ax5.tick_params(axis='x', rotation=45)
    for bar in bars:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}s', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 6. Efficiency Score (middle right)
ax6 = fig.add_subplot(gs[1, 2])
if load_results and file_sizes:
    efficiency_scores = {}
    for fmt in load_results.keys():
        time_score = 1 / (load_results[fmt]['elapsed_time'] + 0.0001)  # Inverted - lower time = higher score
        mem_score = 1 / (abs(load_results[fmt]['memory_delta']) + 0.0001)  # Inverted - lower mem = higher score
        size_score = 1 / (file_sizes.get(fmt, 1) + 0.0001) if fmt in file_sizes else 0
        # Normalize and combine
        combined_score = (time_score * 0.4 + mem_score * 0.3 + size_score * 0.3)
        efficiency_scores[fmt] = combined_score
    
    formats_eff = list(efficiency_scores.keys())
    scores = list(efficiency_scores.values())
    colors_list = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    bars = ax6.bar(formats_eff, scores, color=colors_list[:len(formats_eff)], alpha=0.8, edgecolor='black', linewidth=1.5)
    ax6.set_ylabel('Efficiency Score', fontsize=10, fontweight='bold')
    ax6.set_title('Overall Efficiency', fontsize=11, fontweight='bold')
    ax6.tick_params(axis='x', rotation=45)

# 7. Summary Statistics (bottom)
ax7 = fig.add_subplot(gs[2, :])
ax7.axis('off')

summary_text = "KEY FINDINGS:\n\n"
if 'NetCDF' in load_results and 'Virtual Zarr' in load_results:
    nc_time = load_results['NetCDF']['elapsed_time']
    vz_time = load_results['Virtual Zarr']['elapsed_time']
    improvement = ((nc_time - vz_time) / nc_time * 100)
    summary_text += f"• Virtual Zarr load time is {abs(improvement):.1f}% {'faster' if improvement > 0 else 'slower'} than NetCDF\n"
    
    nc_mem = load_results['NetCDF']['memory_delta']
    vz_mem = load_results['Virtual Zarr']['memory_delta']
    mem_improvement = ((nc_mem - vz_mem) / abs(nc_mem) * 100) if nc_mem != 0 else 0
    summary_text += f"• Virtual Zarr uses {abs(mem_improvement):.1f}% {'less' if mem_improvement > 0 else 'more'} memory than NetCDF\n"

if file_sizes:
    vz_size = file_sizes.get('Virtual Zarr', 0)
    rz_size = file_sizes.get('Regular Zarr', 0)
    if vz_size > 0 and rz_size > 0:
        space_saving = ((rz_size - vz_size) / rz_size * 100)
        summary_text += f"• Virtual Zarr saves {space_saving:.1f}% disk space compared to Regular Zarr\n"

summary_text += "\n✓ All benchmarks completed successfully!"

ax7.text(0.05, 0.95, summary_text, transform=ax7.transAxes, fontsize=11,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n✓ Comprehensive benchmark dashboard generated!")
